In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN attached:", True)
except Exception as e:
    print("HF_TOKEN missing (benchmarks still run):", e)


In [ ]:
!rm -rf /kaggle/working/arc && git clone --branch stage-a-cpt https://github.com/Nyvo2010/arc.git /kaggle/working/arc
!pip install -q -r /kaggle/working/arc/requirements-kaggle.txt


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="jetmoe/jetmoe-8b", local_dir="/kaggle/working/jetmoe-8b")
print("weights ready")


In [ ]:
import glob, json, os
from pathlib import Path
adapter_dirs = {}
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    v = variant.split("_")[0]
    cands = sorted(glob.glob(f"/kaggle/input/**/{v}/adapter", recursive=True))
    cands += sorted(glob.glob("/kaggle/input/arc-tier-b-adapters/{v}/adapter"))
    hits = [c for c in cands if (Path(c) / "adapter_config.json").exists()]
    if hits:
        adapter_dirs[variant] = hits[0]
        print(f"adapter [{variant}] ->{hits[0]}")
    else:
        print(f"!! NO adapter output for [{variant}]")
if not adapter_dirs:
    print("INPUT DIRS:", os.listdir("/kaggle/input"))
    for root in (Path("/kaggle/input")).rglob("adapter_config.json"):
        print("found adapter_config:", root)
    raise SystemExit("No adapters found - is dataset niyuvo/arc-tier-b-adapters attached?")
open("/kaggle/working/adapters.json", "w").write(json.dumps(adapter_dirs, indent=2))


In [ ]:
import json, os, subprocess, sys
adapter_dirs = json.load(open("/kaggle/working/adapters.json"))
adap = ",".join(f"{k}={v}" for k, v in sorted(adapter_dirs.items()))
hard2b = {"bias": 0.38, "k": 12, "halt_threshold": 0.33, "min_gain": 0.07}
os.makedirs("/kaggle/working/calib_final", exist_ok=True)
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    out = f"/kaggle/working/calib_final/calibfinal-{variant}.csv"
    cmd = [sys.executable, "scripts/run_benchmarks.py",
           "--base", "/kaggle/working/jetmoe-8b",
           "--model", variant,
           "--adapters", f"{variant}={adapter_dirs[variant]}",
           "--budgeted", "--max_loops", "4",
           "--tasks", "arc_easy,arc_challenge,hellaswag,piqa,winogrande,boolq,sciq",
           "--limits", "arc_easy=40,arc_challenge=40,hellaswag=40,piqa=40,winogrande=40,boolq=40,sciq=40",
           "--controller", json.dumps(hard2b),
           "--out", out]
    print(">>>", variant, json.dumps(hard2b))
    subprocess.run(cmd, cwd="/kaggle/working/arc", check=True)
# wikitext rows appended per-variant via the same controller
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    out = f"/kaggle/working/calib_final/calibfinal-wt-{variant}.csv"
    cmd = [sys.executable, "scripts/run_benchmarks.py",
           "--base", "/kaggle/working/jetmoe-8b",
           "--model", variant,
           "--adapters", f"{variant}={adapter_dirs[variant]}",
           "--budgeted", "--max_loops", "4",
           "--tasks", "wikitext",
           "--limits", "wikitext=200",
           "--controller", json.dumps(hard2b),
           "--out", out]
    print(">>> wikitext", variant)
    subprocess.run(cmd, cwd="/kaggle/working/arc", check=True)


In [ ]:
import csv, glob, math, os, shutil, json
os.makedirs("/kaggle/output/calibfinal", exist_ok=True)
csvs = sorted(glob.glob("/kaggle/working/calib_final/calibfinal-*.csv"))
for c in csvs:
    shutil.copy(c, f"/kaggle/output/calibfinal/{{os.path.basename(c)}}")
print(f"{{len(csvs)}} final CSVs saved")
mcq = ["arc_easy", "arc_challenge", "hellaswag", "piqa", "winogrande", "boolq", "sciq"]
for key in ["model_adaptive", "block_adaptive", "layer_adaptive"]:
    rows = []
    for c in glob.glob(f"/kaggle/working/calib_final/calibfinal-{{key}}.csv"):
        rows.extend(csv.DictReader(open(c)))
    accs = [float(r["acc"]) * 100 for r in rows if r["task"] in mcq]
    print("\n===", key, "=== avg acc", f"{{sum(accs)/len(accs):.1f}}% over", len(accs), "tasks")
    for r in rows:
        if r["task"] in mcq:
            print(f"  {{r['task']:14s}} acc={{float(r['acc'])*100:5.1f}}  loops={{r['avg_loops_per_item']:>6}}  "
                  f"decide_p={{r.get('avg_decide_mean_p','-'):>6}}  nan={{r['nan_halts']:>3}}  hist={{r.get('halt_hist','')}}")
        else:
            print(f"  wikitext      ppl={{float(r['ppl']):6.1f}}  loops={{r['avg_loops_per_item']:>6}}  tok/s={{r.get('tokens_per_s','-')}}")
